In [ ]:
from functools import lru_cache
from pathlib import Path

import scipy
import numpy as np
import matplotlib.pyplot as plt

import test_discovery
from refnx.reflect import abeles

In [ ]:
# test systems to calculate smeared reflectivity for
unpolarised_systems = ["test4", "test5"]

In [ ]:
# the function for calculating unsmeared reflectivity
# It shouldn't matter which one we use, as long as all the packages agree
fkernel = abeles

# Integrate smearing over (-_INTLIMIT, _INTLIMIT)
_INTLIMIT = 3.5

# Gaussian Quadrature order
QUAD_ORDER = 10001

In [ ]:
@lru_cache(maxsize=128)
def gauss_legendre(n):
    """
    Calculate gaussian quadrature abscissae and weights
    Parameters
    ----------
    n : int
        Gaussian quadrature order.
    Returns
    -------
    (x, w) : tuple
        The abscissae and weights for Gauss Legendre integration.
    """
    return scipy.special.p_roots(n)


def smeared_kernel_pointwise(qvals, w, dqvals, quad_order=17, threads=-1):
    """
    Resolution smearing that uses fixed order Gaussian quadrature integration
    for the convolution.

    Parameters
    ----------
    qvals : array-like
        The Q values for evaluation
    w : array-like
        The uniform slab model parameters in 'layer' form.
    dqvals : array-like
        dQ values corresponding to each value in `qvals`. Each dqval is the
        FWHM of a Gaussian approximation to the resolution kernel.
    quad-order : int, optional
        Specify the order of the Gaussian quadrature integration for the
        convolution.
    threads: int, optional
        Specifies the number of threads for parallel calculation. This
        option is only applicable if you are using the ``_creflect``
        module. The option is ignored if using the pure python calculator,
        ``_reflect``. If `threads == -1` then all available processors are
        used.

    Returns
    -------
    reflectivity : np.ndarray
        The smeared reflectivity
    """
    # get the gauss-legendre weights and abscissae
    abscissa, weights = gauss_legendre(quad_order)

    # get the normal distribution at that point
    prefactor = 1.0 / np.sqrt(2 * np.pi)

    def gauss(x):
        return np.exp(-0.5 * x * x)

    gaussvals = prefactor * gauss(abscissa * _INTLIMIT)

    va = qvals - _INTLIMIT * dqvals
    vb = qvals + _INTLIMIT * dqvals

    assert np.all(va > 0)

    va = va[..., np.newaxis]
    vb = vb[..., np.newaxis]

    qvals_for_res = (abscissa[np.newaxis, :] * (vb - va) + vb + va) / 2.0
    smeared_rvals = fkernel(qvals_for_res, w, threads=threads)

    smeared_rvals *= np.atleast_2d(gaussvals * weights)
    return np.sum(smeared_rvals, -1) * _INTLIMIT

In [ ]:
# grab the layers and data for the systems that need resolution smearing (re-)applied
need_to_smear = {
    td[0].split(".txt")[0]: td
    for td in test_discovery.get_test_data()
    if td[0].split(".txt")[0] in unpolarised_systems
}

In [ ]:
for name, td in need_to_smear.items():
    layers, data = td[1:3]
    qvals = data[:, 0]
    dqvals = data[:, 3]
    r_existing = data[:, 1]

    # calculate smeared data
    r_smeared = smeared_kernel_pointwise(qvals, layers, dqvals, quad_order=QUAD_ORDER)

    try:
        np.testing.assert_allclose(r_existing, r_smeared, rtol=5e-3)
    except AssertionError as e:
        print(name)
        raise e

    data[:, 1] = r_smeared

    pth = Path("..", "test", "unpolarised", "data", f"{name}.dat")
    np.savetxt(pth, data)